# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
ds_meta = dataset.metadata
print(f"Dataset name: {ds_meta.name}")
print(f"Description: {ds_meta.description}")

## 2. Data Overview
Review available Record Sets, Fields, and their IDs.

**Note:** We reference all entities by their `@id` fields for clarity and reproducibility.

In [ ]:
# List all record sets and their fields by `@id`
print("Record Sets and fields by '@id':\n")

record_sets = list(dataset.record_sets)
record_sets_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    record_sets_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the Record Set and Field `@id`s from the overview above.

In [ ]:
# Load records for each Record Set by @id
dataframes = dict()
for rs_id in record_sets_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    # To pandas DataFrame
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(), end="\n\n")
    else:
        print("No records found.\n")

# For continued analysis, select the main record set (the one with most columns and survey data)
if dataframes:
    main_rs_id = max(dataframes, key=lambda x: dataframes[x].shape[1])
    print(f"Using main RecordSet @id for analysis: {main_rs_id}")
    main_df = dataframes[main_rs_id]
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping/categorizing data. All fields and record sets are referenced by their `@id`.

In [ ]:
# Example EDA: Filter, normalize, and group
import numpy as np

if main_rs_id and not main_df.empty:
    # Choose a numeric field and a group field using field @id
    # Let's discover numeric fields:
    rs = next(rs for rs in dataset.record_sets if rs.id == main_rs_id)
    numeric_fields = [f for f in rs.fields if f.data_type in ['Float', 'Integer', 'Number']]
    print(f"Possible numeric fields (@id): {[f.id for f in numeric_fields]}")
    
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id  # e.g., '@id' for GAD-7 score
        print(f"Using numeric field @id: {numeric_field_id}")
        
        # Example threshold: 5 (for depression/anxiety scores)
        threshold = 5
        if numeric_field_id in main_df.columns:
            mask = pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold
            filtered_df = main_df[mask].copy()
            print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
            print(filtered_df[[numeric_field_id]].head())

            # Normalization
            field_series = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
            normalized_col = f"{numeric_field_id}_normalized"
            filtered_df[normalized_col] = (field_series - field_series.mean()) / field_series.std(ddof=0)
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, normalized_col]].head())

            # Group by a categorical field
            # Find candidate group fields (type 'Text', or 'String', likely categorical, not id or score)
            cat_fields = [f for f in rs.fields if f.data_type in ['Text', 'String']]
            group_field_id = cat_fields[0].id if cat_fields else None

            if group_field_id and group_field_id in filtered_df.columns:
                print(f"\nGrouping by field @id: {group_field_id}")
                grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(grouped.head())
            else:
                print("No suitable categorical (group_by) field found in this record set.")
        else:
            print(f"Field {numeric_field_id} not available in data columns.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No records to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Matplotlib is available inline
%matplotlib inline
sns.set(style="whitegrid")

if main_rs_id and not main_df.empty and numeric_fields:
    numeric_field_id = numeric_fields[0].id
    data = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(data.dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Scatter plot for two numeric fields (if available)
    if len(numeric_fields) > 1:
        nf1 = numeric_fields[0].id
        nf2 = numeric_fields[1].id
        plt.figure(figsize=(8,5))
        sns.scatterplot(x=pd.to_numeric(main_df[nf1], errors='coerce'),
                        y=pd.to_numeric(main_df[nf2], errors='coerce'))
        plt.xlabel(nf1)
        plt.ylabel(nf2)
        plt.title(f'Scatter: {nf1} vs. {nf2}')
        plt.show()
else:
    print("Visualization skipped - no numeric fields or data.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- The `mlcroissant` library enables programmatic access and processing of AI-ready survey datasets described by the Croissant schema.
- We loaded and previewed all record sets and fields, referencing entities by their schema `@id`.
- After loading records, basic exploratory data analysis (EDA) and visualization provide quick insights into potential distributions and group differences within the mental health survey.
- For more details about fields and survey specifics, always refer to the dataset's croissant metadata documentation.